<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-04/AI_Modul_4.8_Praktikum_Persiapan_Dataset_untuk_AI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum AI Modul 4.8: Persiapan Dataset untuk AI
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Capaian Pembelajaran Praktikum:
1. Memisahkan matriks fitur ($X$) dan vektor target ($y$) serta menerapkan partisi *Train-Test Split* dan *Stratified K-Fold*.
2. Memahami secara empiris dampak bahaya kebocoran data (*data leakage*) terhadap validitas model AI.
3. Melakukan rekayasa fitur tabular: penskalaan numerik (*StandardScaler*, *RobustScaler*) dan enkoding kategori (*OneHotEncoder*, *OrdinalEncoder*).
4. Menangani ketidakseimbangan kelas ekstrem pada deteksi penyakit tanaman.
5. Membangun alur pipa transformasi otomatis bebas bocor menggunakan *Scikit-Learn ColumnTransformer* dan *Pipeline*.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


In [ ]:
# Membangun dataset perkebunan kelapa sawit realistis (1.000 sampel blok kebun)
np.random.seed(42)
n_samples = 1000

# 1. Fitur numerik normal & kontinu rentan outlier
curah_hujan = np.random.normal(200, 35, n_samples)
# Sisipkan outlier ekstrem pada sensor curah hujan (badai/sensor glitch)
curah_hujan[::50] = curah_hujan[::50] * 3.5 

usia_tanaman = np.random.randint(4, 25, n_samples)
dosis_npk = np.random.uniform(100, 300, n_samples)
tinggi_muka_air = np.random.normal(50, 15, n_samples)

# 2. Fitur kategorikal nominal & ordinal
jenis_tanah = np.random.choice(['Alluvial', 'Gambut', 'Podsolik'], n_samples, p=[0.4, 0.35, 0.25])
topografi = np.random.choice(['Datar', 'Bergelombang', 'Curam'], n_samples, p=[0.6, 0.3, 0.1])
tingkat_gulma = np.random.choice(['Bersih', 'Sedang', 'Padat'], n_samples, p=[0.5, 0.35, 0.15])

# 3. Target Regresi: Tonase Panen (Ton/Ha)
hasil_panen = (
    0.02 * curah_hujan + 
    0.35 * usia_tanaman + 
    0.015 * dosis_npk - 
    0.1 * tinggi_muka_air + 
    np.random.normal(0, 1.5, n_samples)
)

# 4. Target Klasifikasi Imbalance: Infeksi Ganoderma (Hanya ~5% terinfeksi)
prob_sakit = np.clip(0.01 + 0.003 * usia_tanaman + (tinggi_muka_air > 65) * 0.05, 0, 0.4)
status_ganoderma = np.random.binomial(1, prob_sakit)

# Gabungkan ke dalam DataFrame
df_kebun = pd.DataFrame({
    'ID_Blok': [f'BLK-{1000+i}' for i in range(n_samples)],
    'Curah_Hujan_mm': np.round(curah_hujan, 1),
    'Usia_Tanaman_Thn': usia_tanaman,
    'Dosis_NPK_kg': np.round(dosis_npk, 1),
    'Tinggi_Air_Gambut_cm': np.round(tinggi_muka_air, 1),
    'Jenis_Tanah': jenis_tanah,
    'Topografi': topografi,
    'Tingkat_Gulma': tingkat_gulma,
    'Produksi_TBS_Ton': np.round(hasil_panen, 2),
    'Status_Ganoderma': status_ganoderma
})

# Sisipkan sedikit missing values secara acak (data lapangan riil)
df_kebun.loc[10:15, 'Curah_Hujan_mm'] = np.nan
df_kebun.loc[50:53, 'Jenis_Tanah'] = np.nan

print(f"Dataset berhasil dibuat: {df_kebun.shape[0]} baris, {df_kebun.shape[1]} kolom.")
print(f"Distribusi Status Ganoderma: {dict(df_kebun['Status_Ganoderma'].value_counts())}")
df_kebun.head(3)


## 1. Pemisahan Matriks Fitur ($X$) dan Target ($y$) serta Partisi Ilmiah
Langkah pertama sebelum proses analitika adalah mengeliminasi atribut pengenal (*identifiers*) dan memisahkan data menjadi fitur input ($X$) serta target ($y$), dilanjutkan partisi *Train-Test Split* ber-stratifikasi.


In [ ]:
# 1. Eliminasi ID_Blok dan pisahkan fitur prediktor (X) serta target (y)
X = df_kebun.drop(columns=['ID_Blok', 'Produksi_TBS_Ton', 'Status_Ganoderma'])
y_regresi = df_kebun['Produksi_TBS_Ton']
y_klasifikasi = df_kebun['Status_Ganoderma']

# 2. Partisi Train-Test Split ber-stratifikasi pada target klasifikasi
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_klasifikasi,
    test_size=0.20,
    random_state=42,
    stratify=y_klasifikasi
)

print(f"Ukuran Data Latih (X_train): {X_train.shape}")
print(f"Ukuran Data Uji (X_test):   {X_test.shape}")
print(f"Rasio Kelas Ganoderma pada Train: {y_train_cls.mean()*100:.2f}%")
print(f"Rasio Kelas Ganoderma pada Test:  {y_test_cls.mean()*100:.2f}%")


## 2. Demonstrasi Empiris Bahaya Kebocoran Data (*Data Leakage*)
Kebocoran data terjadi jika kita melakukan `fit` (menghitung rata-rata / deviasi standar) pada seluruh dataset sebelum pemisahan train/test. Mari kita buktikan mengapa aturan ini sangat fundamental.


In [ ]:
# Demonstrasi: Perbedaan Nilai Statistik antara Train Saja vs Seluruh Populasi
kolom_uji = 'Dosis_NPK_kg'

mean_populasi = X[kolom_uji].mean()
mean_train_only = X_train[kolom_uji].mean()

std_populasi = X[kolom_uji].std()
std_train_only = X_train[kolom_uji].std()

print("--- Komparasi Statistik Penskalaan ---")
print(f"Mean Seluruh Populasi: {mean_populasi:.4f} | Std: {std_populasi:.4f}")
print(f"Mean Data Latih Saja:   {mean_train_only:.4f} | Std: {std_train_only:.4f}")
print("\nATURAN EMAS: Penskalaan WAJIB menggunakan statistik X_train saja!")


## 3. Penskalaan Fitur Numerik: StandardScaler vs RobustScaler
Variabel curah hujan perkebunan kerap memiliki pencilan akibat cuaca ekstrem. Kita akan membandingkan efektivitas `StandardScaler` (Z-Score) dengan `RobustScaler` (Median-IQR).


In [ ]:
sample_data = X_train[['Curah_Hujan_mm']].dropna().values

scaler_std = StandardScaler()
scaler_rob = RobustScaler()

scaled_std = scaler_std.fit_transform(sample_data)
scaled_rob = scaler_rob.fit_transform(sample_data)

print(f"StandardScaler -> Min: {scaled_std.min():.2f}, Max: {scaled_std.max():.2f}, Mean: {scaled_std.mean():.2f}")
print(f"RobustScaler   -> Min: {scaled_rob.min():.2f}, Max: {scaled_rob.max():.2f}, Median: {np.median(scaled_rob):.2f}")
print("Perhatikan bagaimana RobustScaler menahan lonjakan nilai ekstrem tanpa merusak pemetaan median.")


## 4. Penanganan Ketidakseimbangan Kelas Ekstrem (*Cost-Sensitive Class Weights*)
Mari kita latih model klasifikasi pohon acak (*Random Forest*) untuk mendeteksi infeksi Ganoderma dengan dan tanpa penyesuaian bobot kelas (`class_weight='balanced'`).


In [ ]:
# Siapkan fitur numerik sederhana untuk pengujian klasifikasi
X_num_train = X_train[['Usia_Tanaman_Thn', 'Dosis_NPK_kg', 'Tinggi_Air_Gambut_cm']]
X_num_test = X_test[['Usia_Tanaman_Thn', 'Dosis_NPK_kg', 'Tinggi_Air_Gambut_cm']]

# Model A: Standar tanpa penyesuaian bobot
rf_standar = RandomForestClassifier(random_state=42)
rf_standar.fit(X_num_train, y_train_cls)
y_pred_std = rf_standar.predict(X_num_test)

# Model B: Menggunakan class_weight='balanced'
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_balanced.fit(X_num_train, y_train_cls)
y_pred_bal = rf_balanced.predict(X_num_test)

print("=== Laporan Klasifikasi MODEL STANDAR (Tanpa Pembobotan) ===")
print(classification_report(y_test_cls, y_pred_std, target_names=['Sehat (0)', 'Ganoderma (1)'], zero_division=0))

print("\n=== Laporan Klasifikasi MODEL BALANCED (Class Weight Penalti) ===")
print(classification_report(y_test_cls, y_pred_bal, target_names=['Sehat (0)', 'Ganoderma (1)'], zero_division=0))


## 5. Membangun Pipeline Terpadu Bebas Bocor Menggunakan ColumnTransformer
Sekarang kita integrasikan seluruh komponen ke dalam arsitektur produksi:
1. Imputasi dan penskalaan numerik heterogen.
2. Imputasi dan enkoding kategorikal (nominal dan ordinal).
3. Estimator model prediksi akhir.


In [ ]:
# 1. Kelompokkan kolom berdasarkan perlakuan rekayasa fitur
kolom_num_robust = ['Curah_Hujan_mm', 'Tinggi_Air_Gambut_cm']
kolom_num_standard = ['Usia_Tanaman_Thn', 'Dosis_NPK_kg']
kolom_cat_nominal = ['Jenis_Tanah', 'Topografi']
kolom_cat_ordinal = ['Tingkat_Gulma']

# 2. Definisikan urutan ordinal secara eksplisit
urutan_gulma = [['Bersih', 'Sedang', 'Padat']]

# 3. Susun sub-pipeline untuk masing-masing kelompok fitur
pipe_robust = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

pipe_standard = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

pipe_nominal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

pipe_ordinal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=urutan_gulma))
])

# 4. Gabungkan seluruh sub-pipeline ke dalam ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num_rob', pipe_robust, kolom_num_robust),
    ('num_std', pipe_standard, kolom_num_standard),
    ('cat_nom', pipe_nominal, kolom_cat_nominal),
    ('cat_ord', pipe_ordinal, kolom_cat_ordinal)
])

# 5. Pasang ke dalam Pipeline lengkap bersama estimator Regresi
pipeline_regresi_panen = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Ambil target regresi yang selaras dengan split
_, _, y_train_reg, y_test_reg = train_test_split(
    X, y_regresi, test_size=0.20, random_state=42
)

# Latih pipeline (fit HANYA pada X_train!)
pipeline_regresi_panen.fit(X_train, y_train_reg)

# Prediksi langsung pada data mentah uji X_test
y_pred_reg = pipeline_regresi_panen.predict(X_test)

mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("=== HASIL EVALUASI PIPELINE PRODUKSI BEBAS BOCOR ===")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared Score (R2):     {r2:.4f}")


## 6. Tugas Tantangan Eksplorasi Mandiri (HOTS Challenge)

### Tantangan: Rekayasa Fitur Rasio NPK / Air Gambut
Tambahkan fitur rasio agronomi baru: `Rasio_Hara_Air = Dosis_NPK_kg / (Tinggi_Air_Gambut_cm + 1.0)` secara modular ke dalam DataFrame dan uji apakah performa model regresi meningkat!

### Solusi Tantangan:


In [ ]:
# Salin data latih dan uji untuk rekayasa fitur turunan
X_train_ext = X_train.copy()
X_test_ext = X_test.copy()

X_train_ext['Rasio_Hara_Air'] = X_train_ext['Dosis_NPK_kg'] / (X_train_ext['Tinggi_Air_Gambut_cm'] + 1.0)
X_test_ext['Rasio_Hara_Air'] = X_test_ext['Dosis_NPK_kg'] / (X_test_ext['Tinggi_Air_Gambut_cm'] + 1.0)

# Tambahkan kolom baru ke daftar perlakuan numerik standar
kolom_num_standard_ext = kolom_num_standard + ['Rasio_Hara_Air']

preprocessor_ext = ColumnTransformer(transformers=[
    ('num_rob', pipe_robust, kolom_num_robust),
    ('num_std', pipe_standard, kolom_num_standard_ext),
    ('cat_nom', pipe_nominal, kolom_cat_nominal),
    ('cat_ord', pipe_ordinal, kolom_cat_ordinal)
])

pipeline_ext = Pipeline([
    ('preprocessor', preprocessor_ext),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

pipeline_ext.fit(X_train_ext, y_train_reg)
y_pred_ext = pipeline_ext.predict(X_test_ext)

r2_ext = r2_score(y_test_reg, y_pred_ext)
print(f"R2 Sebelum Rekayasa Fitur Rasio: {r2:.4f}")
print(f"R2 Setelah Rekayasa Fitur Rasio: {r2_ext:.4f}")
print(f"Peningkatan Performa Akurasi:    {(r2_ext - r2)*100:+.2f}%")
